____
### 0. Preamble
This notebook serves as a continuation of my other notebook files `autocomplete.ipynb` and `subqueries.ipynb`. In the past 2 notebooks, the environment was successfully formulated but the model trained was too ineffective. This notebook will attempt to troubleshoot those issues by adjusting the training parameters and the reward shaping of the model 

____
### 1. Imports

In [1]:
from classes.env import SQLEnvAdvanced
from classes.dqn import DQNAgent
import scripts.diagnose as d

#### 1.1 Initialising Objects

In [2]:
env = SQLEnvAdvanced()
agent = DQNAgent(env=env, name="initial-dqn")

In [3]:
agent = agent.load("organic.pth")

c:\Users\xuan2\Desktop\sidequests\launchpad-hackathon-rl-model\classes\agents.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(path, map_location=m

____
### 2. Analysis of performance

In [4]:
counters, rewards = d.diagnose(agent, env, n_episodes=200)

DIAGNOSIS REPORT  (200 episodes)

Reward summary:
  Mean episode reward : -76.23
  Std                 : 31.80
  Min / Max           : -120.00 / -7.40
  Mean episode length : 40.0 steps

Step breakdown (total 8542 steps):
  wrong_illegal          5282  (61.8%)
  correct_block          1566  (18.3%)
  wrong_legal            1152  (13.5%)
  wrong_stop              542  (6.3%)

Stop flag (on correct blocks only):
  correct      1024  (65.4%)
  wrong         542  (34.6%)

Per-step reward distribution:
  reward ~-2.0  → 2071 steps
  reward ~-1.3  → 217 steps
  reward ~-0.3  → 935 steps
  reward ~+2.0  → 758 steps

ROOT CAUSE ANALYSIS:
  ⚠  62% illegal blocks → agent hasn't learned grammar at all.
     Fix: reduce action space, increase learning_starts, check obs encoding.
  ⚠  Episodes hit MAX_TOTAL_STEPS (40.0 avg).
     Agent is not terminating — stop_flag=1 is never being triggered.



- 62% illegal actions
- 0% early termination (all episodes hit the 40-step cap)
- stop_flag correct only 65% of the time

____
### 3. Changes to environment and class
- these are the problems and solutions implemented
1. Flooded learning signal
2. Inaccurate stop flag 
3. Inferred relationship between blocks
4. Overestimation of Q-values

#### 3.1 Flooded learning signal
- at a typical state only 2–6 of the 20 block types are legal, but `act()` argmaxed over all 56 flat actions and random exploration sampled uniformly over all 56. 
    - this means that out of 56 possible blocks, there are only 6 actions that are legal and 1 action that is correct
- with around 90% of actions being illegal, the replay buffer filled up with illegal-move transitions and the `-2.0` illegal penalty drowns out the real learning signal.
    - the model thus fails to learn properly as the replay buffer is filled mostly with penalty
    - even the chances where they reach proper

#### Solution: Action Masking - change to `env.py`
- when deciding an action, the environment checks which blocks are legal and only keep those
- the rest of the blocks have its Q-values set to `-inf` so that the illegal blocks are not considered and the model does not waste moves on illegal blocks
- `legal_action_maks(self)` returns a boolean array of legal blocks for the model to consider
- `legal_mask_from_obs_batch(obs_batch)` does the same thing but does not need a live environment instance 

```python
def legal_action_mask(self):
    legal = legal_next_blocks(self.current_frame.built_seq)
    mask = np.zeros(N_BLOCKS, dtype=bool)
    for b in legal:
        mask[BLOCK_TO_IDX[b]] = True
    return mask
```
- `legal = legal_next_blocks(self.current_frame.built_seq)` : generates an array of syntatically legal next blocks according to the current shape of the array
- `mask = np.zeros(N_BLOCKS, dtype=bool)` : creates a numpy array of boolean, initialised with all false values
- `for b in legal: mask[BLOCK_TO_IDX[b]] = True` : iterates over the array, if the block is legal, changes the value to `true`


```python
def legal_mask_from_obs_batch(obs_batch):
    obs_np = obs_batch.detach().cpu().numpy() if hasattr(obs_batch, "detach") else np.asarray(obs_batch)
    masks = np.zeros((obs_np.shape[0], N_BLOCKS), dtype=bool)
    for i in range(obs_np.shape[0]):
        tokens = obs_np[i, :MAX_SEQUENCE_LEN].astype(int)
        built_seq = [IDX_TO_BLOCK[t - 1] for t in tokens if t > 0]
        legal = legal_next_blocks(built_seq)
        for b in legal:
            masks[i, BLOCK_TO_IDX[b]] = True
    return masks
```
- `obs_np = obs_batch.detach().cpu().numpy() if hasattr(obs_batch, "detach") else np.asarray(obs_batch)` : converts the observation array from a tensor to a numpy array
    - `.detach()` drops it from the autograd graph, `.cpu()` moves it off GPU if needed, `.numpy()` converts it. - `.hasattr(obs_batch, "detach")` check handles both input types safely — if it's already numpy, skip straight to `np.asarray`

- `masks = np.zeros((obs_np.shape[0], N_BLOCKS), dtype=bool)`: initialises a boolean array with all values as false
    - one row of `N_BLOCKS` number of `false` is initialised for every item in the batch    
- `for i in range(obs_np.shape[0]) : tokens = obs_np[i, :MAX_SEQUENCE_LEN].astype(int)` : iterates over each observation in the batch
    - `tokens = obs_np[i, :MAX_SEQUENCE_LEN].astype(int)` grabs the first `MAX_SEQUENCE_LEN` entries which is the encoded built_seq prefix 
    - `built_seq = [IDX_TO_BLOCK[t - 1] for t in tokens if t > 0]` constructs the current sequence in the form of an array of string through indexing the dictionary
    - the next few lines does the exact same thing as `legal_action_mask`



#### 3.2 Inaccurate stop flag
- encoding the action with `action` = `block_idx * 2 + stop_flag` forces the DQN to guess the correct block and the stop flag as one 56-way action
- this can cause the stop flag action to be inaccurate due to the 2 actions being combined into 1

#### Solution: Splitting heads of QNetwork to output 2 heads from a shared base - change to `dqn.py`
```python
self.body = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU())
self.block_head = nn.Linear(hidden, n_blocks)
self.stop_head = nn.Linear(hidden, 2)
```
- `QNetwork` now outputs two heads from a shared trunk:
    - `block_head` — logits over `N_BLOCKS`
    - `stop_head` — binary logits
- Masking is applied only to `block_head`, so the stop decision is never forced to be `-inf` alongside an illegal block.

#### 3.3 Inferred relationship between blocks
- environment's `get_obs()` encodes the built-sequence as integer block indices (0–20) fed straight into a `Linear` layer as floats 
- the NN thus infers that there is correlation between blocks that are close in indexblock 5 is "closer" to block 6 than block 19
- this inferred relationship is meaningless for a symbolic vocabulary dictionary and thus affects the effectiveness of DQN

#### Solution: Using embeddings to eliminate inferred relationship - change to `dqn.py`
```python
self.embed = nn.Embedding(n_blocks + 1, embed_dim)
```
- `QNetwork` now embeds the built-sequence tokens, the `+1` covers the padding tokens
- This also concatenates the flattened embeddings with the rest of the observation (step count, required-blocks vector, depth) before the shared trunk.

#### 3.4 Overestimation of Q-values 
- Vanilla DQN has the tendency to overestimates Q-values
- This overestimation worsens with a wide reward range (`-3` to `+5`) especially when there is no gradient clipping to limit how much the gradient changes during training episodes

#### Solution : Double DQN + gradient clipping - change to `dqn.py`

```python
def dqn_update(self):
    obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size)
    obs = obs.to(self.device)
    actions = actions.to(self.device).long().view(-1)  # flat action = block_idx * 2 + stop_flag
    rewards = rewards.to(self.device).view(-1, 1)
    next_obs = next_obs.to(self.device)
    dones = dones.to(self.device).view(-1, 1)
    block_actions = (actions // 2).view(-1, 1)
    stop_actions = (actions % 2).view(-1, 1)
    next_legal_mask = torch.tensor(legal_mask_from_obs_batch(next_obs), dtype=torch.bool, device=self.device)
    with torch.no_grad():
        next_block_q_online, next_stop_q_online = self.q_net(next_obs)
        next_block_q_online = next_block_q_online.masked_fill(~next_legal_mask, float("-inf"))
        next_block_action = next_block_q_online.argmax(dim=-1, keepdim=True)
        next_stop_action = next_stop_q_online.argmax(dim=-1, keepdim=True)
        next_block_q_tgt, next_stop_q_tgt = self.tgt_q_net(next_obs)
        next_block_q = next_block_q_tgt.gather(1, next_block_action)
        next_stop_q = next_stop_q_tgt.gather(1, next_stop_action)
        max_next_q = next_block_q + next_stop_q  # joint value of the (block, stop) pair
        target_q = rewards + self.gamma * (1.0 - dones) * max_next_q
    block_q, stop_q = self.q_net(obs)
    current_q = block_q.gather(1, block_actions) + stop_q.gather(1, stop_actions)
    loss = nn.functional.mse_loss(current_q, target_q)
    self.q_opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
    self.q_opt.step()
    self.total_updates += 1
    if self.total_updates % self.target_update_every == 0:
            # Polyak/soft update, same style as TD3Agent's target updates
        for p_main, p_tgt in zip(self.q_net.parameters(), self.tgt_q_net.parameters()):
            p_tgt.data.mul_(1.0 - self.tau)
            p_tgt.data.add_(self.tau * p_main.data)
    return loss.item()
```
- Doubl DQN architecture now selects the next action with the online network (block head masked by `legal_mask_from_obs_batch`), evaluates it with the frozen target network — standard Double DQN.
- Combines the two heads' values additively: `Q(s, block, stop) = Q_block(s, block) + Q_stop(s, stop)`.
- `nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)` now clips the gradient before the optimizer step to prevent gradients from changing too much, improving stability.

____
### 4. Importing and training after modifications

In [6]:
from classes.adjenv import SQLEnvAdvanced
from custom.customdqn import DQNAgent as customDQN

#### 4.1 Creating instances of new objects

In [10]:
adjenv = SQLEnvAdvanced()
newagent = customDQN(env=adjenv, name="improved-dqn")

#### 4.2 Training adjusted model in new environment and evaluation

In [5]:
newagent.train(total_timesteps=100_000, learning_starts=200, log_every=50)

Timestep     222 | Episode   50 | Reward:    -1.60 | Epsilon: 0.996
Timestep     467 | Episode  100 | Reward:    -3.50 | Epsilon: 0.991
Timestep     614 | Episode  150 | Reward:    -3.00 | Epsilon: 0.988
Timestep     776 | Episode  200 | Reward:   -10.30 | Epsilon: 0.985
Timestep     955 | Episode  250 | Reward:    -1.00 | Epsilon: 0.982
Timestep    1156 | Episode  300 | Reward:    -1.30 | Epsilon: 0.978
Timestep    1319 | Episode  350 | Reward:    -0.60 | Epsilon: 0.975
Timestep    1500 | Episode  400 | Reward:    -1.00 | Epsilon: 0.972
Timestep    1699 | Episode  450 | Reward:    -3.00 | Epsilon: 0.968
Timestep    1853 | Episode  500 | Reward:    -3.00 | Epsilon: 0.965
Timestep    2104 | Episode  550 | Reward:    -8.10 | Epsilon: 0.960
Timestep    2291 | Episode  600 | Reward:    -3.00 | Epsilon: 0.956
Timestep    2485 | Episode  650 | Reward:    -3.00 | Epsilon: 0.953
Timestep    2775 | Episode  700 | Reward:    -4.50 | Epsilon: 0.947
Timestep    2984 | Episode  750 | Reward:    -3.

In [6]:
results = newagent.evaluate(200)

Episodes: 200
Mean reward: -8.16 +/- 2.92
Reward range: [-9.70, 1.80]
Mean steps: 40.00


#### 4.3 Diagnosis to check performance of new model

In [9]:
counters, rewards = d.diagnose(newagent, adjenv, n_episodes=200)

DIAGNOSIS REPORT  (200 episodes)

Reward summary:
  Mean episode reward : -8.60
  Std                 : 2.53
  Min / Max           : -9.70 / -2.80
  Mean episode length : 40.0 steps

Step breakdown (total 8000 steps):
  wrong_legal            7704  (96.3%)
  correct_block           296  (3.7%)

Stop flag (on correct blocks only):
  correct       296  (100.0%)
  wrong           0  (0.0%)

Per-step reward distribution:
  reward ~-0.3  → 7704 steps
  reward ~+2.0  → 296 steps

ROOT CAUSE ANALYSIS:
  ⚠  96% wrong-but-legal → agent knows grammar but not target.
     Fix: increase required_blocks weight in obs, add shaping reward for
          matching expected block type.
  ⚠  Episodes hit MAX_TOTAL_STEPS (40.0 avg).
     Agent is not terminating — stop_flag=1 is never being triggered.


#### 4.4 Saving the new model

In [ ]:
newagent.save("improved.pth")

____
### 5. Hail Mary
- this section was me trying a hail mary to see how effective a reinforcement learning model can be 
- this model was trained seperately using the following parameters
```python
agent = DQNAgent(env=env, 
                 gamma=0.99,
                 tau = 0.005,
                 lr = 2e-4,
                 batch_size=128, 
                 buffer_capacity=300_000,
                 eps_start=1.0,
                 eps_end = 0.05,
                 eps_decay_steps=700_000,
                 name="final-dqn")
agent.train(total_timestep=1_500_000, learning_starts=8_000, log_every=50)
```
- in addition, the dimensino of the hidden state (`hidden_size`) was also increased from 64 to 128

In [11]:
finalagent = customDQN(env=adjenv, name="final-dqn")
finalagent.load("final.pth")

c:\Users\xuan2\Desktop\sidequests\launchpad-hackathon-rl-model\classes\agents.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(path, map_location=m

In [14]:
results = finalagent.evaluate(200)

Episodes: 200
Mean reward: -7.17 +/- 3.91
Reward range: [-9.70, 4.10]
Mean steps: 40.00


In [16]:
diagnosis = d.diagnose(finalagent, adjenv, 200)

DIAGNOSIS REPORT  (200 episodes)

Reward summary:
  Mean episode reward : -7.03
  Std                 : 4.06
  Min / Max           : -9.70 / +4.10
  Mean episode length : 40.0 steps

Step breakdown (total 8000 steps):
  wrong_legal            7568  (94.6%)
  correct_block           432  (5.4%)

Stop flag (on correct blocks only):
  correct       432  (100.0%)
  wrong           0  (0.0%)

Per-step reward distribution:
  reward ~-0.3  → 7568 steps
  reward ~+2.0  → 432 steps

ROOT CAUSE ANALYSIS:
  ⚠  95% wrong-but-legal → agent knows grammar but not target.
     Fix: increase required_blocks weight in obs, add shaping reward for
          matching expected block type.
  ⚠  Episodes hit MAX_TOTAL_STEPS (40.0 avg).
     Agent is not terminating — stop_flag=1 is never being triggered.


based on this final hail mary attempt:
1. 95% wrong-but-legal → agent knows grammar but not target.
2. Episodes hit MAX_TOTAL_STEPS (40.0 avg).
- this means that the reward shaping has to still be optimised to encourage the model to use the stop flag 
- in addition reward for getting the correct matching block has to be increased so the model receives the proper learning signal to utilise the correct matching block instead of just settling on the legally correct block

However, due to limited time constraints, I will be pivoting to modelling this problem using autoregression, which is a more efficient approach.